<a href="https://colab.research.google.com/github/matheus-haruki/Processamento-de-Linguagem-Natural/blob/main/Aula_12_e_13_Rede_Neural.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Objetivos da aula**

Introdu??o aos conceitos de Redes Neurais Recorrentes (RNNs) e Long Short-Term Memory Networks (LSTMs) e sua aplica??o em tarefas de Processamento de Linguagem Natural

**T?cnicas de PLN implementadas:**


? Implementar uma Rede Neural Recorrente (RNN) simples em Python para prever a pr?xima palavra em uma sequ?ncia de texto, utilizando a biblioteca TensorFlow/Keras.

? Implementar uma Rede Long Short-Term Memory (LSTM) em Python para classificar o sentimento de frases como "positivo" ou "negativo", utilizando a biblioteca TensorFlow/Keras.

**Passo a passo:**

Implementa??o 1: Modelo de Rede Neural de Recorr?ncia



In [ ]:
# Etapa 1: Configura??o do ambiente no Google Colab

#Importar bibliotecas necess?rias

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
print("Bibliotecas importadas com sucesso!")


In [ ]:
# Etapa 2: Prepara??o do Comjunto de Dados

# Conjunto de dados de treinamento (pequeno e simplificado)
textos_treinamento = [
    "eu gosto de programar em python",
    "python ? uma linguagem poderosa",
    "programar ? divertido com python",
    "aprenda python e seja feliz",
    "gosto de aprender coisas novas"
]

print(f"Textos de treinamento: {textos_treinamento}")

In [ ]:
# Inicializar o Tokenizer
tokenizer = Tokenizer()

# Construir o vocabul?rio a partir dos textos
tokenizer.fit_on_texts(textos_treinamento)

# Converter textos em sequ?ncias de n?meros
sequencias = tokenizer.texts_to_sequences(textos_treinamento)

# Imprimir o vocabul?rio e as sequ?ncias geradas
print(f"\nvocabul?rio (palavra: ?ndice): {tokenizer.word_index}")
print(f"Sequ?ncias num?ricas dos textos: {sequencias}")

# Calcular o tamanho do vocabul?rio (+1 para incluir o 0 de padding)
total_palavras = len(tokenizer.word_index) + 1
print(f"Tamanho total do vocabul?rio: {total_palavras}")

In [ ]:
# Preparar Entradas (X) e Sa?das (y) para a previs?o da pr?xima palavra
# a entrada (X) ser? uma sequ?ncia de palavras, e a sa?da (y) ser? a palavra seguinte.
# Determinar o comprimento m?ximo das sequ?ncias para padding
max_comprimento = max(len(seq) for seq in sequencias)
print(f"\nComprimento m?ximo das sequ?ncias antes do padding: {max_comprimento}")

# Criar pares de entrada (sequ?ncia parcial) e sa?da (pr?xima palavra)
# Ex: "eu gosto de programar" -> "em"
# "gosto de programar em" -> "python"
entradas_X = []
sa?das_y = []

for seq in sequencias:
    for i in range(1, len(seq)):
        entradas_X.append(seq[:i])  # A sequ?ncia at? a palavra atual
        sa?das_y.append(seq[i])     # A pr?xima palavra

print(f"Exemplo de entradas_X (parcial): {entradas_X[0:5]}")
print(f"Exemplo de sa?das_y (parcial): {sa?das_y[0:5]}")

# Padronizar o comprimento das sequ?ncias de entrada
# Todas as sequ?ncias de entrada precisam ter o mesmo comprimento para a RNN
entradas_X_padded = pad_sequences(entradas_X, maxlen=max_comprimento - 1, padding='pre')
# O maxlen ? 'max_comprimento - 1' porque a sa?da 'y' ? a ?ltima palavra, ent?o X sempre ter? 1 palavra a menos.

# Converter as sa?das para o formato one-hot encoding
# Isso ? necess?rio para a camada de sa?da da RNN (softmax)
sa?das_y_one_hot = tf.keras.utils.to_categorical(sa?das_y, num_classes=total_palavras)

print(f"\nExemplo de entradas_X_padded (ap?s padding e truncagem): \n{entradas_X_padded[0:5]}")
print(f"Exemplo de sa?das_y_one_hot (ap?s one-hot encoding): \n{sa?das_y_one_hot[0:5]}")
print(f"Formato final das entradas (X): {entradas_X_padded.shape}")
print(f"Formato final das sa?das (y): {sa?das_y_one_hot.shape}")

In [ ]:
# Etapa 3: Constru??o do Modelo RNN

# 1. Definindo o modelo
# Definir a arquitetura do modelo RNN
modelo_rnn = Sequential()

# Camada de Embedding:
# total_palavras: tamanho do vocabul?rio
# 10: dimens?o do vetor de embedding (quantas caracter?sticas queremos para cada palavra)
# input_length: comprimento das sequ?ncias de entrada (maxlen - 1)
modelo_rnn.add(Embedding(total_palavras, 10, input_length=entradas_X_padded.shape[1]))

# Camada SimpleRNN:
# 32: n?mero de unidades (neur?nios) na camada recorrente. Este ? o tamanho do estado oculto.
modelo_rnn.add(SimpleRNN(32))

# Camada Densa de Sa?da:
# total_palavras: n?mero de neur?nios de sa?da (um para cada palavra no vocabul?rio)
# activation='softmax': fun??o de ativa??o para probabilidade (soma 1 para todas as palavras)
modelo_rnn.add(Dense(total_palavras, activation='softmax'))

# Compilar o modelo
modelo_rnn.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Exibir um resumo da arquitetura do modelo
modelo_rnn.summary()

In [ ]:
# Etapa 4: Treinamento do Modelo

# Treinando o modelo
print("\nIniciando o treinamento do modelo RNN...")
modelo_rnn.fit(entradas_X_padded, sa?das_y_one_hot, epochs=100, verbose=1)
   # epochs: quantas vezes o modelo ver? todo o conjunto de dados
   # verbose: 1 para mostrar o progresso do treinamento
print("Treinamento conclu?do!")

In [ ]:
# Etapa 5: Usar o modelo para previs?o

# 1. Fun??o de Previs?o
def prever_proxima_palavra(modelo, tokenizer, max_seq_len, texto_base):
    """
    Prev? a pr?xima palavra dado um texto base.
    """
    # Converter o texto base para sequ?ncia num?rica
    sequencia_numerica = tokenizer.texts_to_sequences([texto_base])[0]

    # Padronizar o comprimento da sequ?ncia de entrada (como no treinamento)
    sequencia_padded = pad_sequences([sequencia_numerica], maxlen=max_seq_len, padding='pre')

    # Fazer a previs?o
    previsao_probabilidades = modelo.predict(sequencia_padded, verbose=0)[0]

    # Obter o ?ndice da palavra com maior probabilidade
    indice_palavra_prevista = np.argmax(previsao_probabilidades)

    # Converter o ?ndice de volta para a palavra
    for palavra, indice in tokenizer.word_index.items():
        if indice == indice_palavra_prevista:
            return palavra
    return None  # Caso a palavra n?o seja encontrada

# Comprimento de entrada esperado pelo modelo
comprimento_entrada_modelo = entradas_X_padded.shape[1]

# Testar o modelo com algumas frases
print("\n-- Testando o Modelo RNN --")

texto_teste_1 = "eu gosto de"
proxima_1 = prever_proxima_palavra(modelo_rnn, tokenizer, comprimento_entrada_modelo, texto_teste_1)
print(f"Texto: '{texto_teste_1}' -> Pr?xima palavra prevista: '{proxima_1}'")

texto_teste_2 = "python ? uma"
proxima_2 = prever_proxima_palavra(modelo_rnn, tokenizer, comprimento_entrada_modelo, texto_teste_2)
print(f"Texto: '{texto_teste_2}' -> Pr?xima palavra prevista: '{proxima_2}'")

texto_teste_3 = "programar ? divertido"
proxima_3 = prever_proxima_palavra(modelo_rnn, tokenizer, comprimento_entrada_modelo, texto_teste_3)
print(f"Texto: '{texto_teste_3}' -> Pr?xima palavra prevista: '{proxima_3}'")

texto_teste_4 = "aprenda python e"
proxima_4 = prever_proxima_palavra(modelo_rnn, tokenizer, comprimento_entrada_modelo, texto_teste_4)
print(f"Texto: '{texto_teste_4}' -> Pr?xima palavra prevista: '{proxima_4}'")

# Exemplo com palavras fora do vocabul?rio
texto_teste_5 = "o sol brilha no"  # Palavras "sol" e "brilha" podem n?o estar no vocabul?rio
proxima_5 = prever_proxima_palavra(modelo_rnn, tokenizer, comprimento_entrada_modelo, texto_teste_5)
print(f"Texto: '{texto_teste_5}' -> Pr?xima palavra prevista: '{proxima_5}' (pode ser inesperada devido a palavras desconhecidas)")


**Implementa??o 2: Modelo de Rede Neural Rede Long Short-Term Memory**

In [ ]:
# Etapa 1: Configura??o do Ambiente e Importa??o de Bibliotecas

# Important bibliotecas necess?rias
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

print("Bibliotecas importadas com sucesso!")

In [ ]:
# Etapa 2: Prepara??o do conjunto de dados e an?lise de sentimentos

# Definir o Conjunto de Dados (Frases e R?tulos) para an?lise de sentimentos
dados_sentimento = [
 ("este filme ? ?timo e divertido", "positivo"),
 ("eu adorei o livro, muito bom", "positivo"),
 ("gostei muito da atua??o dos atores", "positivo"),
 ("o roteiro ? fraco e chato", "negativo"),
 ("n?o recomendo este p?ssimo produto", "negativo"),
 ("uma perda de tempo horr?vel", "negativo"),
 ("?timo trabalho, parab?ns", "positivo"),
 ("terr?vel experi?ncia, nunca mais", "negativo"),
 ("excelente servi?o, muito eficiente", "positivo"),
 ("que decep??o, muito ruim", "negativo"),
 ("aprendizagem de m?quina ? fascinante", "positivo"),
 ("pln ? um campo interessante", "positivo"),
 ("este software travou v?rias vezes", "negativo"),
 ("a interface ? confusa e dif?cil", "negativo"),
 ("o aplicativo ? super ?til e r?pido", "positivo"),
]

textos = [dado[0] for dado in dados_sentimento]
sentimentos = [dado[1] for dado in dados_sentimento]

print(f"Total de frases: {len(textos)}")
print(f"Exemplo de textos: {textos[:3]}")
print(f"Exemplo de sentimentos: {sentimentos[:3]}")

In [ ]:
# Mapear Sentimentos para N?meros: converter ?positivo? e ?negativo? para 0 e 1
mapeamento_sentimento = {'negativo': 0, 'positivo': 1}

rotulos_numericos = np.array([mapeamento_sentimento[s] for s in sentimentos])

print(f"\nSentimentos mapeados para n?meros: {rotulos_numericos}")


In [ ]:
# Tokeniza??o de Texto
tokenizer = Tokenizer(num_words=None, oov_token="")  # oov_token define o token para palavras desconhecidas
tokenizer.fit_on_texts(textos)
sequencias_numericas = tokenizer.texts_to_sequences(textos)

total_palavras_vocab = len(tokenizer.word_index) + 1  # +1 para o ?ndice 0 (usado no padding e OOV)

print(f"\nVocabul?rio (palavra: ?ndice): {tokenizer.word_index}")
print(f"Sequ?ncias num?ricas das frases: {sequencias_numericas}")
print(f"Tamanho total do vocabul?rio: {total_palavras_vocab}")

# Padronizar o comprimento das sequ?ncias
max_len = max(len(s) for s in sequencias_numericas)
print(f"\nComprimento m?ximo das sequ?ncias: {max_len}")

sequencias_padded = pad_sequences(sequencias_numericas, maxlen=max_len, padding='post')
print(f"Sequ?ncias ap?s padding: \n{sequencias_padded}")

# Dividir os dados em conjuntos de treinamento e teste
from sklearn.model_selection import train_test_split

X_treino, X_teste, y_treino, y_teste = train_test_split(
    sequencias_padded,
    rotulos_numericos,
    test_size=0.2,
    random_state=42,
    stratify=rotulos_numericos
)

print(f"\nShape de X_treino: {X_treino.shape}")
print(f"Shape de X_teste: {X_teste.shape}")
print(f"Shape de y_treino: {y_treino.shape}")
print(f"Shape de y_teste: {y_teste.shape}")


In [ ]:
# Etapa 3: Constru??o do Modelo LSTM

#Definir a arquitetura do modelo LSTM
modelo_lstm = Sequential()

# Camada de Embedding: Converte os ?ndices num?ricos das palavras em vetores densos.
# total_palavras_vocab: tamanho do vocabul?rio
# 50: dimens?o do vetor de embedding (pode ser ajustado)
# input_length: comprimento padronizado das sequ?ncias (max_len)
modelo_lstm.add(Embedding(total_palavras_vocab, 50, input_length=max_len))

# Camada LSTM:
# 64: n?mero de unidades (neur?nios) na camada LSTM. Define o tamanho do estado oculto e da c?lula de mem?ria.
# dropout: Um tipo de regulariza??o para evitar overfitting (descarta aleatoriamente neur?nios durante o treinamento).
# recurrent_dropout: Dropout aplicado nas conex?es recorrentes da LSTM.
modelo_lstm.add(LSTM(64, dropout=0.2, recurrent_dropout=0.2))

# Camada Densa de Sa?da:
# 1: Um ?nico neur?nio de sa?da, pois ? um problema de classifica??o bin?ria (positivo/negativo).
# activation='sigmoid': Fun??o de ativa??o para classifica??o bin?ria (produz um valor entre 0 e 1).
modelo_lstm.add(Dense(1, activation='sigmoid'))

# Compilar o modelo
modelo_lstm.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Exibir um resumo da arquitetura do modelo
modelo_lstm.summary()

In [ ]:
# Etapa 4: Treinamento e avalia??o do Modelo

# Treinar o modelo
print("\nIniciando o treinamento do modelo LSTM...")
historico = modelo_lstm.fit(
    X_treino, y_treino,
    epochs=50, # Reduzi para 50 epochs para um treinamento mais r?pido no exemplo. Pode ser aumentado.
    batch_size=2, # Pequeno batch_size para dataset pequeno.
    validation_split=0.1, # Usar 10% do treino para valida??o
    verbose=1
)
# epochs: n?mero de vezes que o modelo ver? todo o conjunto de dados de treinamento.
# batch_size: n?mero de amostras por atualiza??o de gradiente.
# validation_split: % dos dados de treino usados para valida??o durante o treinamento (opcional, mas bom para monitorar overfitting).
print("treinamento conclu?do!")

In [ ]:
# Avalia o modelo no conjunto de teste
perda, acuracia = modelo_lstm.evaluate(X_teste, y_teste, verbose=0)
print(f"\nAcur?cia do modelo no conjunto de teste: {acuracia * 100:.2f}%")
print(f"Perda do modelo no conjunto de teste: {perda:.4f}")

# Fazer previs?es no conjunto de teste
y_pred_prob = modelo_lstm.predict(X_teste)
y_pred_classes = (y_pred_prob > 0.5).astype(int)  # Converter probabilidades para 0 ou 1

print("\n--- Relat?rio de Classifica??o ---")
print(classification_report(y_teste, y_pred_classes, target_names=['negativo', 'positivo']))

print("\n--- Matriz de Confus?o ---")
cm = confusion_matrix(y_teste, y_pred_classes)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['negativo', 'positivo'],
            yticklabels=['negativo', 'positivo'])
plt.xlabel('Previsto')
plt.ylabel('Real')
plt.title('Matriz de Confus?o')
plt.show()


In [ ]:
# Etapa 5: Testar o modelo com novas frases

# Utilizando o modelo treinado
def prever_sentimento(modelo, tokenizer, max_seq_len, frase_nova, mapeamento_sentimento):
    """
    Prev? o sentimento de uma nova frase.
    """
    # Converter a frase para sequ?ncia num?rica
    sequencia_numerica = tokenizer.texts_to_sequences([frase_nova])

    # Verificar se a frase tem apenas palavras desconhecidas
    if not sequencia_numerica or not sequencia_numerica[0]:
        print(f"Aviso: A frase '{frase_nova}' cont?m apenas palavras desconhecidas.")
        return "Desconhecido"

    sequencia_numerica = sequencia_numerica[0]  # Pega a primeira (e ?nica) sequ?ncia

    # Padronizar o comprimento da sequ?ncia de entrada
    sequencia_padded = pad_sequences([sequencia_numerica], maxlen=max_seq_len, padding='post')

    # Fazer a previs?o (probabilidade)
    probabilidade_positiva = modelo.predict(sequencia_padded, verbose=0)[0][0]

    # Inverter o mapeamento para obter o nome do sentimento
    mapeamento_inverso = {v: k for k, v in mapeamento_sentimento.items()}

    # Classificar com base no limiar de 0.5
    if probabilidade_positiva >= 0.5:
        return mapeamento_inverso[1]  # 'positivo'
    else:
        return mapeamento_inverso[0]  # 'negativo'


# Testar o modelo com novas frases
print("\n--- Testando o Modelo LSTM com Novas Frases ---")

frases = [
    "gostei muito do filme, excelente!",
    "odiei o livro, muito entediante",
    "a aula de pln ? ?tima",
    "o atendimento foi p?ssimo",
    "esse produto n?o vale a pena, ? caro",
    "o filme ? legal",
    "isso ? horr?vel, que tristeza"
]

for i, frase in enumerate(frases, 1):
    sentimento = prever_sentimento(modelo_lstm, tokenizer, max_len, frase, mapeamento_sentimento)
    print(f"Frase {i}: '{frase}' -> Sentimento previsto: '{sentimento}'")
